# Estaciones del IDEAM · precipitación, temperatura y humedad (2023–2024)
Notebook aparte del principal (`descarga_datos_v2.ipynb`), porque la descarga desde datos.gov.co es lenta: muchas consultas pequeñas, una por estación y año.

**Antes:** ejecuta las secciones 1–3 del notebook principal, que crean `datos/procesados/plantas_solares.gpkg`. Usa las mismas carpetas `datos/` y `resultados/`, así que guárdalo en la misma carpeta que el principal.

**Después:** vuelve a correr la sección 5 del principal y la tabla panel incluirá la lluvia, los días de lluvia, la temperatura y la humedad de la estación más cercana.

**Para que sea más rápido:**
- `MAX_EST_POR_PLANTA` limita cuántas estaciones cercanas se consultan por planta (por defecto 3).
- `VARIABLES_IDEAM` permite quitar las variables que no necesites.
- Lo descargado se guarda: si se interrumpe, al volver a correr solo pide lo que falta.

## 1. Preparación y parámetros

In [ ]:
import sys, subprocess, time
from datetime import date
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "requests", "geopandas", "matplotlib"], check=True)
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

INICIO, FIN = date(2023, 1, 1), date(2024, 12, 31)
RADIO_EST_KM = 30          # estaciones a menos de esta distancia de cada planta
MAX_EST_POR_PLANTA = 3     # solo las N más cercanas a cada planta (None = todas las del radio)
UMBRAL_LLUVIA_MM = 1.0     # un día cuenta como lluvioso si llueve al menos esto
MIN_DIAS_MES = 20          # días con dato que debe tener una estación para aceptar su valor mensual
DESCARGAR = {"ideam_estaciones": True}

BASE = Path.cwd()
RAW, PROC, OUT = BASE / "datos" / "crudos", BASE / "datos" / "procesados", BASE / "resultados"
for p in (RAW, PROC, OUT):
    p.mkdir(parents=True, exist_ok=True)
f_pl = PROC / "plantas_solares.gpkg"
if not f_pl.exists():
    raise FileNotFoundError("Falta datos/procesados/plantas_solares.gpkg: ejecuta primero las secciones 1–3 de descarga_datos_v2.ipynb")
plantas = gpd.read_file(f_pl)
plantas["codigo_xm"] = plantas["codigo_xm"].astype(str)
print(f"{len(plantas)} plantas")

## 2. Catálogo de estaciones y estaciones cercanas a cada planta
Datos medidos en tierra, publicados por el IDEAM en **Datos Abiertos** (datos.gov.co, API Socrata, sin cuenta):

| Conjunto | Variable | Paso original | Se guarda como |
|---|---|---|---|
| `hp9r-jxuu` | Catálogo Nacional de Estaciones (ubicación, altitud, categoría, estado) | — | tabla de estaciones |
| `s54a-sgyg` | Precipitación (mm) | 10 min (automáticas) / diaria | suma diaria → lluvia mensual y **días de lluvia** |
| `sbwg-7ju4` | Temperatura del aire a 2 m (°C) | 10 min – horaria | media, máxima y mínima diaria → mensual |
| `uext-mhny` | Humedad relativa del aire a 2 m (%) | 10 min – horaria | media diaria → mensual |

- Solo se consultan las estaciones a menos de `RADIO_EST_KM` de alguna planta.
- La agregación diaria la hace el servidor (`$group` de SoQL), así que no se descargan millones de registros de 10 minutos.
- Un mes de una estación se acepta si tiene al menos `MIN_DIAS_MES` días con dato.

**Para qué sirven:** son la única medición directa en tierra. Permiten **validar** la lluvia de NASA POWER y CHIRPS, contar los **días de lluvia** reales cerca de cada planta, y aportar la **temperatura** (los paneles pierden eficiencia con el calor) y la **humedad** (asociada a nubosidad baja). Para agregar otra variable, añade su identificador de datos.gov.co a `VARIABLES_IDEAM`.

In [ ]:
SOCRATA = "https://www.datos.gov.co/resource/{}.json"
VARIABLES_IDEAM = {   # nombre: (identificador en datos.gov.co, agregaciones diarias)
    "precipitacion": ("s54a-sgyg", {"valor": "sum"}),
    "temperatura":   ("sbwg-7ju4", {"valor": "avg", "vmax": "max", "vmin": "min"}),
    "humedad":       ("uext-mhny", {"valor": "avg"}),
}

def socrata(ident, params, lote=50000, intentos=3, espera=180):
    """Consulta paginada a la API de datos.gov.co. Si la API rechaza la consulta, muestra su mensaje."""
    filas, offset = [], 0
    while True:
        for k in range(intentos):
            try:
                r = requests.get(SOCRATA.format(ident), params=params | {"$limit": lote, "$offset": offset}, timeout=espera)
            except requests.RequestException as e:          # red caída o tiempo agotado → reintenta
                if k == intentos - 1:
                    raise RuntimeError(f"sin respuesta de datos.gov.co: {e}")
                time.sleep(10); continue
            if r.status_code == 200:
                break
            if r.status_code < 500 or k == intentos - 1:     # 4xx: la consulta está mal, no sirve reintentar
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
            time.sleep(10)
        bloque = r.json()
        filas += bloque
        if len(bloque) < lote:
            return pd.DataFrame(filas)
        offset += lote

def distancia_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# 1) Catálogo de estaciones y estaciones cercanas a cada planta
f_cat = RAW / "ideam_catalogo_estaciones.csv"
if DESCARGAR["ideam_estaciones"] and not f_cat.exists():
    socrata("hp9r-jxuu", {}).drop(columns=["ubicaci_n"], errors="ignore").to_csv(f_cat, index=False)
cat = pd.read_csv(f_cat, dtype={"codigo": str})
cat["latitud"], cat["longitud"] = pd.to_numeric(cat["latitud"], errors="coerce"), pd.to_numeric(cat["longitud"], errors="coerce")
cat = cat.dropna(subset=["latitud", "longitud"])

cerca = []
for _, p in plantas.iterrows():
    d = distancia_km(p.lat, p.lon, cat.latitud.values, cat.longitud.values)
    sel = cat.loc[d <= RADIO_EST_KM, ["codigo", "nombre", "categoria", "tecnologia", "estado", "altitud", "latitud", "longitud"]].copy()
    sel["distancia_km"] = d[d <= RADIO_EST_KM].round(1)
    if MAX_EST_POR_PLANTA:
        sel = sel.nsmallest(MAX_EST_POR_PLANTA, "distancia_km")
    cerca.append(sel.assign(codigo_xm=p.codigo_xm))
est_plantas = pd.concat(cerca, ignore_index=True).sort_values(["codigo_xm", "distancia_km"])
est_plantas.to_csv(PROC / "ideam_estaciones_por_planta.csv", index=False)
CODIGOS_EST = sorted(est_plantas.codigo.unique())
print(f"{len(cat)} estaciones en el catálogo · {len(CODIGOS_EST)} a menos de {RADIO_EST_KM} km de alguna planta")
est_plantas.groupby("codigo_xm").size().rename("estaciones_cercanas")

## 3. Series diarias (una estación y un año por consulta)

In [ ]:
# 2) Series diarias de cada variable, una estación y un año a la vez (consultas pequeñas)
COLS_DIARIO = lambda aggs: ["codigoestacion", "dia", *aggs, "n_obs"]

def consulta_diaria(ident, aggs, codigo, t0, t1, convertir=True):
    v = "valorobservado::number" if convertir else "valorobservado"
    sel = ", ".join(f"{f}({v}) as {k}" for k, f in aggs.items())
    return socrata(ident, {
        "$select": f"codigoestacion, date_trunc_ymd(fechaobservacion) as dia, {sel}, count(valorobservado) as n_obs",
        "$where": (f"codigoestacion = '{codigo}' AND "
                   f"fechaobservacion between '{t0:%Y-%m-%d}T00:00:00' and '{t1:%Y-%m-%d}T23:59:59'"),
        "$group": "codigoestacion, dia", "$order": "dia"})

def bloque_diario(ident, aggs, codigo, t0, t1):
    """Intenta el periodo completo; si falla, prueba sin conversión de tipo y luego mes a mes."""
    for convertir in (True, False):
        try:
            return [consulta_diaria(ident, aggs, codigo, t0, t1, convertir)]
        except Exception as e:
            ultimo = e
    partes = []
    for m in pd.period_range(t0, t1, freq="M"):
        try:
            partes.append(consulta_diaria(ident, aggs, codigo, m.start_time, m.end_time))
        except Exception:
            pass
    if not partes:
        raise ultimo
    return partes

def diario_ideam(ident, aggs, codigos):
    partes, omitidas = [], []
    años = [(pd.Timestamp(max(INICIO, date(a, 1, 1))), pd.Timestamp(min(FIN, date(a, 12, 31)))) for a in range(INICIO.year, FIN.year + 1)]
    for n, c in enumerate(codigos, 1):
        for t0, t1 in años:
            try:
                partes += bloque_diario(ident, aggs, c, t0, t1)
            except Exception as e:
                omitidas.append(c); print(f"   {c} {t0.year} omitida: {str(e)[:150]}")
            time.sleep(0.5)
        if n % 10 == 0:
            print(f"   {n}/{len(codigos)} estaciones")
    partes = [d for d in partes if len(d)]
    if not partes:
        return pd.DataFrame(columns=COLS_DIARIO(aggs))
    df = pd.concat(partes, ignore_index=True)
    df["dia"] = pd.to_datetime(df["dia"])
    df = df.drop_duplicates(["codigoestacion", "dia"])
    for k in list(aggs) + ["n_obs"]:
        df[k] = pd.to_numeric(df.get(k), errors="coerce")
    if omitidas:
        print(f"   {len(set(omitidas))} estaciones con algún año omitido")
    return df[COLS_DIARIO(aggs)]

if not CODIGOS_EST:
    print(f"⚠️ No hay estaciones del IDEAM a menos de {RADIO_EST_KM} km de las plantas: aumenta RADIO_EST_KM.")
diarios = {}
for nombre, (ident, aggs) in VARIABLES_IDEAM.items():
    f = RAW / f"ideam_{nombre}_diaria.csv"
    if DESCARGAR["ideam_estaciones"] and not f.exists() and CODIGOS_EST:
        print("IDEAM", nombre, f"… ({len(CODIGOS_EST)} estaciones)")
        try:
            d = diario_ideam(ident, aggs, CODIGOS_EST)
            if len(d):
                d.to_csv(f, index=False)          # solo se guarda si llegó algo: si no, se reintenta la próxima vez
        except Exception as e:
            print(f"   {nombre} no se pudo descargar: {e}")
    d = pd.read_csv(f, dtype={"codigoestacion": str}) if f.exists() else pd.DataFrame(columns=COLS_DIARIO(aggs))
    d["dia"] = pd.to_datetime(d["dia"])
    diarios[nombre] = d
    print(f"   {nombre}: {d.codigoestacion.nunique()} estaciones con datos, {len(d)} días-estación")

## 4. Valores mensuales por estación y capa de puntos

In [ ]:
# 3) Valores mensuales por estación
def mensual_estacion(df, columnas):
    if df.empty:
        return pd.DataFrame(columns=["codigoestacion", "mes", "dias_con_dato", *columnas])
    df = df.assign(mes=df.dia.dt.to_period("M").astype(str))
    g = df.groupby(["codigoestacion", "mes"])
    out = pd.DataFrame({"dias_con_dato": g.size()})
    for nuevo, (col, f) in columnas.items():
        out[nuevo] = g[col].agg(f)
    out = out.reset_index()
    out.loc[out.dias_con_dato < MIN_DIAS_MES, list(columnas)] = np.nan
    return out

pp = diarios["precipitacion"].assign(lluvioso=lambda d: d.valor >= UMBRAL_LLUVIA_MM)
est_mes = (mensual_estacion(pp, {"ideam_ppt_mm": ("valor", "sum"), "ideam_dias_lluvia": ("lluvioso", "sum"),
                                 "ideam_ppt_max_diaria_mm": ("valor", "max")})
           .merge(mensual_estacion(diarios["temperatura"], {"ideam_temp_media_c": ("valor", "mean"),
                                   "ideam_temp_max_c": ("vmax", "mean"), "ideam_temp_min_c": ("vmin", "mean")})
                  .drop(columns="dias_con_dato"), on=["codigoestacion", "mes"], how="outer")
           .merge(mensual_estacion(diarios["humedad"], {"ideam_hr_media_pct": ("valor", "mean")})
                  .drop(columns="dias_con_dato"), on=["codigoestacion", "mes"], how="outer"))
num = est_mes.columns.difference(["codigoestacion", "mes"])
est_mes[num] = est_mes[num].apply(pd.to_numeric, errors="coerce")
est_mes.to_csv(PROC / "ideam_estaciones_mensual.csv", index=False)

# Capa de puntos (tipo punto) con el promedio de cada estación en el periodo
prom = est_mes.groupby("codigoestacion").mean(numeric_only=True).reset_index()
est_gdf = cat.merge(prom, left_on="codigo", right_on="codigoestacion")
est_gdf = gpd.GeoDataFrame(est_gdf, geometry=gpd.points_from_xy(est_gdf.longitud, est_gdf.latitud), crs=4326)
est_gdf.to_file(PROC / "ideam_estaciones.gpkg", driver="GPKG")
print(f"{est_gdf.codigo.nunique()} estaciones con datos mensuales")
est_gdf[["codigo", "nombre", "categoria", "ideam_ppt_mm", "ideam_dias_lluvia", "ideam_temp_media_c", "ideam_hr_media_pct"]].round(1)

## 5. Mapa de las estaciones

In [ ]:
dptos = gpd.read_file(PROC / "departamentos.gpkg")
lo0, la0, lo1, la1 = (-79.1, -4.3, -66.8, 13.5)
FIG = (4, 5.8)
def base(ax, relleno=False):
    if relleno:
        dptos.plot(ax=ax, color="#F2F2EF", edgecolor="none", zorder=0)
    dptos.boundary.plot(ax=ax, color="0.45", linewidth=0.3, zorder=2)
    ax.set_xlim(lo0, lo1); ax.set_ylim(la0, la1); ax.set_aspect("equal"); ax.set_axis_off()
def barra(fig, ax, m, etiqueta):
    cb = fig.colorbar(m, ax=ax, orientation="horizontal", fraction=0.04, pad=0.01, aspect=30)
    cb.set_label(etiqueta, fontsize=10); cb.ax.tick_params(labelsize=8)
def guardar(fig, nombre):
    (OUT / "mapas_presentacion").mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT / "mapas_presentacion" / nombre, dpi=200, bbox_inches="tight", facecolor="white"); plt.show()

# Mapa · Estaciones del IDEAM cerca de las plantas (lluvia media anual observada)
e = est_gdf.dropna(subset=["ideam_ppt_mm"]).copy()
e["ppt_anual"] = e["ideam_ppt_mm"] * 12
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
m = ax.scatter(e.geometry.x, e.geometry.y, s=28, c=e.ppt_anual, cmap="YlGnBu", vmin=0, vmax=4000,
               edgecolor="k", linewidth=0.4, zorder=3)
plantas.plot(ax=ax, color="#E0A526", marker="^", markersize=22, edgecolor="k", zorder=4)
barra(fig, ax, m, "Lluvia media anual en estaciones IDEAM (mm)"); guardar(fig, "estaciones_ideam.png")